# A/B Test Analysis — Recommender System

This notebook performs statistical significance testing on A/B experiment data,
computing CTR lift, add-to-cart conversion lift, and confidence intervals.

**Targets:**
- CTR lift ≥ 10%
- Add-to-cart conversion lift ≥ 5%
- Statistical significance: p < 0.05 (two-sided z-test for proportions)
- Cold-start failure rate ≤ 12%

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path().resolve().parent))
from src.config import ARTIFACTS_DIR

print('Libraries loaded')

## 1. Load or Simulate Event Data

In production this reads from the `event_log` PostgreSQL table.
Here we simulate realistic A/B data to demonstrate the analysis pipeline.

In [ ]:
rng = np.random.default_rng(42)

N_USERS    = 3000   # users per variant
CTR_CTRL   = 0.082  # control click-through rate
CTR_TREAT  = 0.097  # treatment CTR (lift ~18%)
ATC_CTRL   = 0.031  # add-to-cart rate (control)
ATC_TREAT  = 0.034  # add-to-cart (treatment, ~10% lift)

def simulate_variant(n, ctr, atc_rate, variant_name):
    impressions = rng.integers(8, 20, size=n)  # per-user impressions
    clicks      = rng.binomial(impressions, ctr)
    add_to_cart = rng.binomial(clicks, atc_rate / ctr)
    return pd.DataFrame({
        'user_id':      np.arange(n) + (0 if variant_name == 'control' else n),
        'variant':      variant_name,
        'impressions':  impressions,
        'clicks':       clicks,
        'add_to_cart':  add_to_cart,
    })

control_df   = simulate_variant(N_USERS, CTR_CTRL, ATC_CTRL, 'control')
treatment_df = simulate_variant(N_USERS, CTR_TREAT, ATC_TREAT, 'treatment')
ab_df = pd.concat([control_df, treatment_df], ignore_index=True)

print(ab_df.groupby('variant')[['impressions','clicks','add_to_cart']].sum())

## 2. CTR Lift — Two-proportion Z-test

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

def ab_ztest(successes_ctrl, trials_ctrl, successes_treat, trials_treat, metric_name):
    """Two-proportion z-test + lift + 95% confidence interval."""
    rate_ctrl  = successes_ctrl  / trials_ctrl
    rate_treat = successes_treat / trials_treat
    lift       = (rate_treat - rate_ctrl) / rate_ctrl

    stat, p = proportions_ztest(
        count=[successes_treat, successes_ctrl],
        nobs=[trials_treat, trials_ctrl],
        alternative='larger',
    )

    ci_low_c,  ci_hi_c  = proportion_confint(successes_ctrl,  trials_ctrl,  alpha=0.05)
    ci_low_t,  ci_hi_t  = proportion_confint(successes_treat, trials_treat, alpha=0.05)

    print(f'\n── {metric_name} ─────────────────────────────')
    print(f'  Control   : {rate_ctrl:.4f}  95% CI [{ci_low_c:.4f}, {ci_hi_c:.4f}]')
    print(f'  Treatment : {rate_treat:.4f}  95% CI [{ci_low_t:.4f}, {ci_hi_t:.4f}]')
    print(f'  Lift      : {lift:+.1%}')
    print(f'  Z-stat    : {stat:.3f}')
    print(f'  p-value   : {p:.4f}  → {"✓ SIGNIFICANT" if p < 0.05 else "✗ NOT SIGNIFICANT"}')
    return {'metric': metric_name, 'rate_ctrl': rate_ctrl, 'rate_treat': rate_treat,
            'lift': lift, 'z_stat': stat, 'p_value': p, 'significant': p < 0.05}

agg = ab_df.groupby('variant')[['impressions','clicks','add_to_cart']].sum()

ctr_result = ab_ztest(
    agg.loc['control','clicks'],    agg.loc['control','impressions'],
    agg.loc['treatment','clicks'],  agg.loc['treatment','impressions'],
    'CTR (Click-Through Rate)',
)

atc_result = ab_ztest(
    agg.loc['control','add_to_cart'],    agg.loc['control','clicks'],
    agg.loc['treatment','add_to_cart'],  agg.loc['treatment','clicks'],
    'Add-to-Cart Conversion',
)

## 3. Cold-Start Failure Rate

In [ ]:
# Cold-start = users who received popularity fallback instead of personalised recs
# Simulating: ~8% of users are cold-start in treatment
n_served       = N_USERS
n_cold_start   = int(N_USERS * 0.08)
cold_start_pct = n_cold_start / n_served

TARGET_COLD_START = 0.12

print(f'Cold-start users : {n_cold_start:,} / {n_served:,}')
print(f'Cold-start rate  : {cold_start_pct:.1%}  (target ≤ {TARGET_COLD_START:.0%})')
print(f'Target met       : {"✓" if cold_start_pct <= TARGET_COLD_START else "✗"}')

## 4. Success Metrics Summary

In [ ]:
results = [
    {'Metric': 'CTR Lift',           'Target': '≥ 10%',   'Achieved': f"{ctr_result['lift']:+.1%}",  'Status': '✓' if ctr_result['lift'] >= 0.10 else '✗'},
    {'Metric': 'Add-to-Cart Lift',   'Target': '≥ 5%',    'Achieved': f"{atc_result['lift']:+.1%}",  'Status': '✓' if atc_result['lift'] >= 0.05 else '✗'},
    {'Metric': 'Cold-Start Failure', 'Target': '≤ 12%',   'Achieved': f"{cold_start_pct:.1%}",       'Status': '✓' if cold_start_pct <= 0.12 else '✗'},
    {'Metric': 'CTR p-value',        'Target': '< 0.05',  'Achieved': f"{ctr_result['p_value']:.4f}",'Status': '✓' if ctr_result['significant'] else '✗'},
    {'Metric': 'ATC p-value',        'Target': '< 0.05',  'Achieved': f"{atc_result['p_value']:.4f}",'Status': '✓' if atc_result['significant'] else '✗'},
]

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

## 5. Optional — Load from PostgreSQL

Replace the simulation above with this cell when the database is available:

```python
import sqlalchemy
engine = sqlalchemy.create_engine(os.getenv('DATABASE_URL'))

ab_df = pd.read_sql("""
    SELECT a.variant,
           COUNT(CASE WHEN e.event_type = 'impression'  THEN 1 END) AS impressions,
           COUNT(CASE WHEN e.event_type = 'click'       THEN 1 END) AS clicks,
           COUNT(CASE WHEN e.event_type = 'add_to_cart' THEN 1 END) AS add_to_cart
    FROM recommender.ab_assignments a
    JOIN recommender.event_log e ON e.user_id = a.user_id
                                AND e.experiment_id = a.experiment_id
    WHERE a.experiment_id = 'exp_hybrid_vs_svd'
    GROUP BY a.variant
""", engine)
```